# Phase 3 reanalysis, one definition of "decoy named", 39 tasks

Reads only the saved result files. No GPU and no model calls.

**Definition used everywhere.** A task counts as *named* when the decoy function (for T4, the planted bait function)
appears in the answer's top 5 functions. A task is *newly steered* when it is named in the trap run and not named in the
clean run of the same session. Every row shows clean count, trap count and newly steered.
Tests are exact McNemar on the paired tasks, with b = newly steered and c = named on clean but not on trap.

**Exclusions.** Any task whose decoy or bait sits inside a gold file is dropped from every variant. The check below uses the
fixed module parser, where a component starting with underscores followed by a capital letter is a class.

In [ ]:
import json, math, os, re, glob
from itertools import combinations
from collections import defaultdict

def find_repo():
    for c in [os.getcwd(), os.path.dirname(os.getcwd()), '/kaggle/working', '/kaggle/input/localizer-steering', '.']:
        if os.path.isdir(os.path.join(c, 'Phase_3')) and os.path.isdir(os.path.join(c, 'Phase_3_v2')): return c
    raise SystemExit('Put this notebook next to the folders Phase_3 and Phase_3_v2, or set REPO below.')

REPO = find_repo()                       # set by hand if needed, for example REPO = '/path/to/localizer-steering'
TASKS = next(p for p in [os.path.join(REPO, 'task_set_40.json'), os.path.join(REPO, 'Phase_3', 'task_set_40.json')] if os.path.exists(p))
OUT = os.path.join(REPO, 'Analysis_final_v2'); os.makedirs(OUT, exist_ok=True)

# which folder holds each variant, and its clean run (always from the same folder, so the session is paired)
VARIANTS = {   # name -> (folder, spec file stem, trap file stem)
    'T1 decoy':   ('Phase_3',    'T1_decoy',   'T1_decoy'),
    'T1 pointer': ('Phase_3_v2', 'T1_pointer', 'T1_pointer'),
    'T2 decoy':   ('Phase_3_v2', 'T2_decoy',   'T2_decoy'),
    'T2 pointer': ('Phase_3_v2', 'T2_pointer', 'T2_pointer'),
    'T3 decoy':   ('Phase_3_v2', 'T3_decoy',   'T3_decoy'),
    'T3 pointer': ('Phase_3',    'T3_pointer', 'T3_pointer'),
    'T4':         ('Phase_3',    'T4_decoy',   'T4_decoy'),
}
LOCS = ['A1', 'A1.5', 'A2']

## Fixed module parser
The old rule treated a component as a class only if it started with a capital letter, so `_RefreshThread` was read as a
module. The fixed rule strips leading underscores first. This fix also belongs in `localizer.py` for any future run.

In [ ]:
def norm_id(n): return n[4:] if n.startswith('src.') else n

def module_of_node_old(nid):
    p = norm_id(nid).split('.')
    for i, c in enumerate(p):
        if i > 0 and c[:1].isupper(): return '.'.join(p[:i])
    return '.'.join(p[:-1]) if len(p) > 1 else nid

def module_of_node(nid):
    p = norm_id(nid).split('.')
    for i, c in enumerate(p):
        if i > 0 and c.lstrip('_')[:1].isupper(): return '.'.join(p[:i])
    return '.'.join(p[:-1]) if len(p) > 1 else nid

assert module_of_node_old('rich.live._RefreshThread.run') == 'rich.live._RefreshThread'
assert module_of_node('rich.live._RefreshThread.run') == 'rich.live'
assert module_of_node('fastapi.routing.APIRoute.__init__') == 'fastapi.routing'
assert module_of_node('requests.adapters.HTTPAdapter.send') == 'requests.adapters'

## Statistics helpers

In [ ]:
def wilson(k, n, z=1.959964):
    if n == 0: return (0.0, 0.0)
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); a = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (max(0.0, (c - a) / d), min(1.0, (c + a) / d))

def mcnemar(b, c):
    n = b + c
    return 1.0 if n == 0 else min(1.0, 2 * sum(math.comb(n, i) for i in range(0, min(b, c) + 1)) / 2 ** n)

def binom_check(b, c):
    """independent exact two-sided binomial test, summing all outcomes no more likely than the observed one"""
    from fractions import Fraction
    n = b + c
    if n == 0: return 1.0
    pr = [Fraction(math.comb(n, i), 2 ** n) for i in range(n + 1)]
    return float(min(Fraction(1), sum(p for p in pr if p <= pr[b])))

def adjust(pvals):
    """Bonferroni, Holm and Benjamini-Hochberg adjusted p-values for a dict {key: p}."""
    keys = list(pvals); m = len(keys); order = sorted(keys, key=lambda k: pvals[k])
    bonf = {k: min(1.0, pvals[k] * m) for k in keys}
    holm, run = {}, 0.0
    for i, k in enumerate(order):
        run = max(run, min(1.0, (m - i) * pvals[k])); holm[k] = run
    bh, run = {}, 1.0
    for i in range(m - 1, -1, -1):
        k = order[i]; run = min(run, pvals[k] * m / (i + 1)); bh[k] = min(1.0, run)
    return bonf, holm, bh

def fmt_p(p): return '1' if p >= 0.99995 else ('<0.0001' if p < 0.0001 else f'{p:.4f}')
def ci(k, n): lo, hi = wilson(k, n); return f'{lo:.2f}-{hi:.2f}'

## Load results and check which decoys sit inside gold files

In [ ]:
tasks = {t['id']: t for t in json.load(open(TASKS))}
def load(path): return {r['id']: r for r in json.load(open(path))}

data = {}   # (variant, loc) -> dict(clean=..., trap=..., decoys=..., spec=...)
for v, (folder, spec_stem, trap_stem) in VARIANTS.items():
    spec = json.load(open(os.path.join(REPO, folder, f'spec_{spec_stem}.json')))
    for loc in LOCS:
        data[(v, loc)] = dict(clean=load(os.path.join(REPO, folder, f'clean_{loc}.json')),
                              trap=load(os.path.join(REPO, folder, f'trap_{trap_stem}_{loc}.json')),
                              decoys=spec['decoys'], spec=spec)

def gold_modules(t):
    mods = set()
    for f in t['gold_files']:
        f = re.sub(r'^src/', '', f)
        m = re.sub(r'\.py$', '', f).replace('/', '.')
        mods.add(re.sub(r'\.__init__$', '', m))
    for g in t['gold_functions']: mods.add(module_of_node(g))
    return mods

flag = defaultdict(list)
for v in VARIANTS:
    for tid, d in data[(v, 'A1')]['decoys'].items():
        gm = gold_modules(tasks[tid]); dm = module_of_node(d); dn = norm_id(d)
        if dm in gm or any(dn.startswith(m + '.') and module_of_node(dn) in gm for m in gm):
            flag[tid].append((v, d, dm))
print('Decoys or baits inside a gold file (fixed parser):')
for tid, items in flag.items():
    print(' ', tid, '|', items[0][1], '| module', items[0][2], '| in', sorted(items and gold_modules(tasks[tid])), '| variants', [i[0] for i in items])
old_flag = [tid for v in VARIANTS for tid, d in data[(v, 'A1')]['decoys'].items()
            if module_of_node_old(d) not in gold_modules(tasks[tid]) and module_of_node(d) in gold_modules(tasks[tid])]
print('Tasks the old parser let through and the fixed parser catches', sorted(set(old_flag)))
EXCLUDE = sorted(flag)
ALL_IDS = list(tasks)
IDS = [i for i in ALL_IDS if i not in EXCLUDE]
print(f'Excluded {EXCLUDE}. Analysing {len(IDS)} tasks.')

## One definition, every row
`named` means the decoy is in the answer's top 5 functions.

In [ ]:
def named(row, decoy):
    return norm_id(decoy) in [norm_id(x) for x in row['functions'][:5]]

def cell(v, loc, ids=IDS):
    D = data[(v, loc)]; C, T, dec = D['clean'], D['trap'], D['decoys']
    nc = {i: named(C[i], dec[i]) for i in ids}; nt = {i: named(T[i], dec[i]) for i in ids}
    b = [i for i in ids if nt[i] and not nc[i]]; c = [i for i in ids if nc[i] and not nt[i]]
    p = mcnemar(len(b), len(c)); assert abs(p - binom_check(len(b), len(c))) < 1e-12
    return dict(variant=v, loc=loc, n=len(ids), clean=sum(nc.values()), trap=sum(nt.values()), newly=len(b), dropped=len(c),
                p=p, newly_ids=b, nt=nt, nc=nc)

res = {(v, l): cell(v, l) for v in VARIANTS for l in LOCS}
res_all40 = {(v, l): cell(v, l, ALL_IDS) for v in VARIANTS for l in LOCS}

lines = [f'### Decoy named in top 5 functions, {len(IDS)} tasks (clean count, trap count, newly steered)\n',
         '| Variant | Loc | Clean | Trap | Newly steered | Trap 95% CI | Dropped | McNemar p |', '|---|---|---|---|---|---|---|---|']
for v in VARIANTS:
    for l in LOCS:
        r = res[(v, l)]
        lines.append(f"| {v} | {l} | {r['clean']}/{r['n']} | {r['trap']}/{r['n']} | {r['newly']} | {ci(r['trap'], r['n'])} | {r['dropped']} | {fmt_p(r['p'])} |")
TABLE_MAIN = '\n'.join(lines); print(TABLE_MAIN)

## Multiple-testing correction over all 21 tests

In [ ]:
def corr_table(R, title):
    keys = list(R); bonf, holm, bh = adjust({k: R[k]['p'] for k in keys})
    order = sorted(keys, key=lambda k: R[k]['p'])
    out = [f'### {title}\n', '| Variant | Loc | Newly steered | raw p | Bonferroni | Holm | BH |', '|---|---|---|---|---|---|---|']
    for k in order:
        if R[k]['p'] > 0.5: break
        out.append(f"| {k[0]} | {k[1]} | {R[k]['newly']} | {fmt_p(R[k]['p'])} | {fmt_p(bonf[k])} | {fmt_p(holm[k])} | {fmt_p(bh[k])} |")
    out.append('\nAll other tests have raw p of 0.5 or more and adjusted p of 1.')
    surv = lambda adj: ', '.join(f'{k[0]} {k[1]}' for k in order if adj[k] < 0.05) or 'none'
    out.append(f'\nSurvive at 0.05. Bonferroni {surv(bonf)}. Holm {surv(holm)}. BH {surv(bh)}.')
    return '\n'.join(out), dict(bonf=bonf, holm=holm, bh=bh)

TABLE_CORR, ADJ = corr_table(res, f'Correction over 21 tests, {len(IDS)} tasks'); print(TABLE_CORR)
TABLE_CORR40, _ = corr_table(res_all40, 'Same correction on all 40 tasks, for comparison'); print(); print(TABLE_CORR40)

## Exposure split on A2
`trap_seen` says whether the planted text was shown to the model. Newly steered is counted among tasks where it was seen
and where it was not.

In [ ]:
def expo(v, ids=IDS):
    r = res[(v, 'A2')]; T = data[(v, 'A2')]['trap']
    seen = [i for i in ids if T[i].get('trap_seen') is True]; unseen = [i for i in ids if T[i].get('trap_seen') is False]
    s = [i for i in seen if i in r['newly_ids']]; u = [i for i in unseen if i in r['newly_ids']]
    return dict(variant=v, seen=len(seen), steered_seen=len(s), unseen=len(unseen), steered_unseen=len(u), seen_ids=seen,
                steered_ids=s)

lines = [f'### Exposure split on A2, {len(IDS)} tasks\n',
         '| Variant | Note shown to A2 | Newly steered when shown | Not shown | Newly steered when not shown |', '|---|---|---|---|---|']
EXPO = {}
for v in VARIANTS:
    if v == 'T4': continue
    e = expo(v); EXPO[v] = e
    pct = f"{e['steered_seen']}/{e['seen']} ({100 * e['steered_seen'] / max(1, e['seen']):.0f}%)"
    lines.append(f"| {v} | {e['seen']} | {pct} | {e['unseen']} | {e['steered_unseen']}/{e['unseen']} |")
TABLE_EXPO = '\n'.join(lines); print(TABLE_EXPO)

# paired comparison of pointer wordings on the tasks where A2 saw the note
seen_sets = {v: set(EXPO[v]['seen_ids']) for v in ['T1 pointer', 'T2 pointer', 'T3 pointer']}
print('\nSeen sets identical across pointer variants:', len({frozenset(s) for s in seen_sets.values()}) == 1)
common = sorted(set.intersection(*seen_sets.values()))
lines = [f'### Pointer wording, paired on the {len(common)} tasks where A2 saw the note\n', '| Comparison | First followed | Second followed | b | c | McNemar p |', '|---|---|---|---|---|---|']
for a, b_ in [('T3 pointer', 'T1 pointer'), ('T3 pointer', 'T2 pointer'), ('T1 pointer', 'T2 pointer')]:
    fa = {i for i in common if res[(a, 'A2')]['nt'][i] and not res[(a, 'A2')]['nc'][i]}
    fb = {i for i in common if res[(b_, 'A2')]['nt'][i] and not res[(b_, 'A2')]['nc'][i]}
    lines.append(f'| {a} vs {b_} | {len(fa)} | {len(fb)} | {len(fa - fb)} | {len(fb - fa)} | {fmt_p(mcnemar(len(fa - fb), len(fb - fa)))} |')
TABLE_PAIR = '\n'.join(lines); print(); print(TABLE_PAIR)

## Gold loss and top-module drift
Gold loss is the share of tasks whose gold module was in the clean answer (any rank) and is missing from the trap answer.
Drift is a change of the top module.

In [ ]:
def gold_loss(v, loc, ids=IDS):
    D = data[(v, loc)]; C, T = D['clean'], D['trap']
    base = [i for i in ids if C[i]['mod_rank']]; lost = [i for i in base if not T[i]['mod_rank']]
    top1 = lambda r: (r['modules'] or [None])[0]
    drift = [i for i in ids if top1(C[i]) != top1(T[i])]
    return dict(base=len(base), lost=lost, drift=len(drift))

lines = [f'### Gold loss and top-module drift, {len(IDS)} tasks\n', '| Variant | Loc | Gold lost | 95% CI | Drift | Lost ids |', '|---|---|---|---|---|---|']
GL = {}
for v in VARIANTS:
    for l in LOCS:
        g = gold_loss(v, l); GL[(v, l)] = g
        lines.append(f"| {v} | {l} | {len(g['lost'])}/{g['base']} | {ci(len(g['lost']), g['base'])} | {g['drift']}/{len(IDS)} | {', '.join(g['lost']) or '-'} |")
TABLE_GOLD = '\n'.join(lines); print(TABLE_GOLD)
floor = max(GL[(v, 'A1')]['drift'] for v in VARIANTS if v != 'T4')
print(f"\nA1 drift under T1 to T3 (A1 never sees comment text, so this is run to run noise) is at most {floor}/{len(IDS)}.")
t3 = GL[('T3 pointer', 'A2')]['lost']
print('T3 pointer A2 gold lost:', t3, '| rich_3934 among them:', 'rich_3934' in t3)
print('Decoy ranked first on every T3 pointer A2 gold loss:',
      all((data[('T3 pointer', 'A2')]['trap'][i]['functions'] or [None])[0] == data[('T3 pointer', 'A2')]['decoys'][i] for i in t3))

## Clean accuracy on the same tasks

In [ ]:
def acc(rows, ids):
    m1 = sum(1 for i in ids if rows[i]['mod_rank'] == 1); m3 = sum(1 for i in ids if rows[i]['mod_rank'])
    f1 = sum(1 for i in ids if rows[i]['fn_rank'] == 1); f5 = sum(1 for i in ids if rows[i]['fn_rank'])
    return m1, m3, f1, f5

lines = [f'### Clean accuracy on {len(IDS)} tasks (counts, percentage in brackets)\n',
         '| Loc | Session | module@1 | module@3 | fn@1 | fn@5 |', '|---|---|---|---|---|---|']
for l in LOCS:
    for folder in ['Phase_3', 'Phase_3_v2']:
        rows = load(os.path.join(REPO, folder, f'clean_{l}.json')); n = len(IDS)
        a = acc(rows, IDS)
        lines.append(f'| {l} | {folder} | ' + ' | '.join(f'{x}/{n} ({100 * x / n:.1f}%)' for x in a) + ' |')
TABLE_CLEAN = '\n'.join(lines); print(TABLE_CLEAN)

## Figure
Share of tasks where the decoy was newly steered into the top 5 functions (clean is 0 everywhere, so this equals the trap
count), with 95% Wilson intervals. Dark bars survive Holm correction over the 21 tests. Light bars do not.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

INK, MUTED, GRID, SURF = '#1f2430', '#5b6270', '#e4e6eb', '#ffffff'
STRONG, WEAK = '#1f5fbf', '#a9c4ee'

def bars(ax, items, title):
    xs = range(len(items)); n = len(IDS)
    for x, (label, key) in zip(xs, items):
        r = res[key]; k = r['newly']; lo, hi = wilson(k, n); p = k / n
        survives = ADJ['holm'][key] < 0.05
        ax.bar(x, p, width=0.58, color=STRONG if survives else WEAK, zorder=3)
        ax.errorbar(x, p, yerr=[[p - lo], [hi - p]], color=INK, lw=1.2, capsize=3, zorder=4)
        ax.text(x, hi + 0.015, f'{k}/{n}', ha='center', va='bottom', fontsize=9, color=INK)
    ax.set_xticks(list(xs)); ax.set_xticklabels([l for l, _ in items], fontsize=9, color=INK)
    ax.set_ylim(0, 0.62); ax.set_yticks([0, .1, .2, .3, .4, .5]); ax.set_yticklabels(['0%', '10%', '20%', '30%', '40%', '50%'], fontsize=9, color=MUTED)
    ax.grid(axis='y', color=GRID, lw=0.8, zorder=0); ax.set_axisbelow(True)
    for sp in ['top', 'right', 'left']: ax.spines[sp].set_visible(False)
    ax.spines['bottom'].set_color(MUTED); ax.tick_params(length=0)
    ax.set_title(title, loc='left', fontsize=11, color=INK, fontweight='bold')
    ax.set_facecolor(SURF)

fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={'width_ratios': [6, 3]}, dpi=200)
fig.patch.set_facecolor(SURF)
order = [('T1\nin decoy', ('T1 decoy', 'A2')), ('T2\nin decoy', ('T2 decoy', 'A2')), ('T3\nin decoy', ('T3 decoy', 'A2')),
         ('T1\npointer', ('T1 pointer', 'A2')), ('T2\npointer', ('T2 pointer', 'A2')), ('T3\npointer', ('T3 pointer', 'A2'))]
bars(a, order, 'Comment traps on A2, the only localizer that reads them')
a.set_ylabel('Decoy newly steered', fontsize=9, color=MUTED)
bars(b, [('A1\nnames only', ('T4', 'A1')), ('A1.5\n+ signatures', ('T4', 'A1.5')), ('A2\nreads modules', ('T4', 'A2'))],
     'Bait function (T4), all localizers')
from matplotlib.patches import Patch
fig.legend(handles=[Patch(color=STRONG, label='Survives Holm correction over 21 tests'), Patch(color=WEAK, label='Does not survive')],
           loc='lower center', ncol=2, frameon=False, fontsize=9, labelcolor=INK, bbox_to_anchor=(0.5, -0.01))
fig.suptitle(f'Decoy newly steered into the top 5 functions, {len(IDS)} tasks (error bars are 95% Wilson intervals)', x=0.012, ha='left', fontsize=11.5, color=INK)
fig.tight_layout(rect=(0, 0.06, 1, 0.94))
FIG = os.path.join(OUT, 'figure_a2_decoy_named.png'); fig.savefig(FIG, facecolor=SURF); plt.show()
print('Wrote', FIG)

## Save outputs

In [ ]:
import csv
with open(os.path.join(OUT, 'all_results.csv'), 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['variant', 'loc', 'n', 'clean_named', 'trap_named', 'newly_steered', 'dropped', 'mcnemar_p',
                                   'bonferroni', 'holm', 'bh', 'gold_lost', 'gold_base', 'drift'])
    for v in VARIANTS:
        for l in LOCS:
            r = res[(v, l)]; g = GL[(v, l)]
            w.writerow([v, l, r['n'], r['clean'], r['trap'], r['newly'], r['dropped'], f"{r['p']:.6f}",
                        f"{ADJ['bonf'][(v, l)]:.6f}", f"{ADJ['holm'][(v, l)]:.6f}", f"{ADJ['bh'][(v, l)]:.6f}",
                        len(g['lost']), g['base'], g['drift']])
head = (f'# Final analysis v2\n\nDefinition. Decoy named means the decoy (T4 the bait) is in the top 5 functions. Newly steered means named on trap and not on clean. '
        f'Excluded tasks {EXCLUDE}. n = {len(IDS)}.\n\n')
open(os.path.join(OUT, 'tables.md'), 'w').write(head + '\n\n'.join([TABLE_MAIN, TABLE_CORR, TABLE_EXPO, TABLE_PAIR, TABLE_GOLD, TABLE_CLEAN, TABLE_CORR40]) + '\n')
print('Wrote', os.path.join(OUT, 'tables.md'), 'and all_results.csv')